In [ ]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
import TCP_IP_UTILS
import Utils
import time

In [ ]:
#Add power supply code here -> DC Sources
import sys
import pyvisa
sys.path.append(r"../PowerSupply")
import PS_Utils
Device_ID_DC = "USB0::0x2A8D::0x1202::MY59003914::0::INSTR"
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
        print(" -", r)
device_dc = PS_Utils.connect_E36313A(Device_ID_DC)
DIG_LS = 1
ANA_LS = 2
MAIN = 3

In [ ]:
rm = pyvisa.ResourceManager()
OSC_ID = 'USB0::0x2A8D::0x9007::MY62310119::0::INSTR'
scope = rm.open_resource(OSC_ID)  # Use your actual VISA address
# Optional: Set timeout and chunk size
scope.timeout = 5000  # in ms
scope.chunk_size = 1024000  # for large data transfers

# Identify instrument
print("IDN:", scope.query("*IDN?"))

In [ ]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

In [ ]:
#First Turn on Digital Leval Shifters
PS_Utils.set_voltage_E36313A(device_dc,DIG_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,DIG_LS))
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

In [ ]:
#Turn on Main Chip PS
PS_Utils.set_voltage_E36313A(device_dc,MAIN,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,MAIN))

In [ ]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_LM["id"]
scan_len = Utils.DL_EN_SC_LM["len"]
scan_data = np.zeros(scan_len,dtype=int).tolist()
#scan_data[4] = 1
#scan_data[1] = 1
#scan_data[4] = 1
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

In [ ]:
#Call DL EN Scan Chain here with data
scan_id = Utils.DL_EN_SC_RM["id"]
scan_len = Utils.DL_EN_SC_RM["len"]
scan_data = np.zeros(scan_len,dtype=int).tolist()
#scan_data[1] = 1
#scan_data[4] = 1
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

In [ ]:
#Control Logic TD Data
import numpy as np
import random
import re
def scan_gen(time):
    SC_data = f"""
    IN_EXT_LOC = 1'b0;
    TDC_EN_EXT_LOC = 1'b0;
    ENCHG_EXT_LOC = 1'b1;
    ENTIME_EXT_LOC = 1'b1;
    PCHG_EXT_LOC = 1'b0;
    VDAC_CTRL_EXT_LOC = 1'b1;
    DEL_RST_EXT_LOC = 1'b0;
    TDC_COMPUTE_EXT_LOC = 1'b0;
    TDC_RST_EXT_LOC = 1'b0;
    VTC_EN_EXT_LOC = 1'b1;

    IN_LB = 6'd3;
    IN_UB = 6'd22; 
    TDC_EN_LB = 6'd1;
    TDC_EN_UB = 6'd21;
    ENCHG_LB = 6'd0;
    ENCHG_UB = 6'd0;
    ENTIME_LB = 6'd0;
    ENTIME_UB = 6'd0;
    PCHG_LB = 6'd2;
    PCHG_UB = 6'd22;
    VDAC_CTRL_LB = 6'd0;
    VDAC_CTRL_UB = 6'd0; 
    DEL_RST_LB = 6'd2;
    DEL_RST_UB = 6'd22;
    TDC_COMPUTE_LB = 6'd19;
    TDC_COMPUTE_UB = 6'd24;
    TDC_RST_LB = 6'd1;
    TDC_RST_UB = 6'd21;
    BL_NUM_TGT = 2'd3;
    BL_DONE_TIME = 6'd50;
    VTC_EN_LB = 6'd0;
    VTC_EN_UB = 6'd0;
    SAMPLE_EDGE_TIME = 6'd{time};

    CHG_TIME = 1'd0;
    OSC_DEL = 1'd0;
    """
    SC_order = ["BL_DONE_TIME","SAMPLE_EDGE_TIME", "BL_NUM_TGT", "BLANK_4", "IN_LB", "TDC_EN_LB", "ENCHG_LB", "IN_EXT_LOC","TDC_EN_EXT_LOC","ENCHG_EXT_LOC",\
                "ENTIME_EXT_LOC","PCHG_EXT_LOC","VDAC_CTRL_EXT_LOC", 'ENTIME_LB', 'PCHG_LB', 'VDAC_CTRL_LB', 'DEL_RST_LB', 'TDC_COMPUTE_LB', 'TDC_RST_LB', \
                'VTC_EN_LB',"DEL_RST_EXT_LOC","TDC_COMPUTE_EXT_LOC","TDC_RST_EXT_LOC","VTC_EN_EXT_LOC","CHG_TIME","OSC_DEL", "VTC_EN_UB","TDC_RST_UB","TDC_COMPUTE_UB",\
                "DEL_RST_UB","VDAC_CTRL_UB","PCHG_UB","ENTIME_UB","ENCHG_UB","TDC_EN_UB","IN_UB","BLANK_12_PISO"]

    lines = SC_data.splitlines()
    sc_signal = {}
    for line in lines:
        if line.strip():  # Skip empty lines
            # Handle 'd' (decimal) format
            match = re.match(r"(\w+) = (\d+)'d([0-9]+);", line.strip())
            if match:
                signal_name = match.group(1)
                num_bits = int(match.group(2))  # Number of bits
                signal_value = match.group(3)  # Decimal value


                # If there's only 1 bit, do not add the bit-range
                if num_bits == 1:
                    name = signal_name
                else:
                    # Format the name with the bit-range (e.g., VTC_EN_UB<[5:0]>)
                    name = "%s" % (signal_name) # Using 0-based index for the range

                # Format the decimal value to binary and pad to the correct number of bits
                binary_value = bin(int(signal_value))[2:].zfill(num_bits)
                sc_signal[name] = binary_value


            # Handle 'b' (binary) format
            elif "b" in line:  # e.g., BL_NUM_TGT = 2'b01;
                match = re.match(r"(\w+) = (\d+)'b([01]+);", line.strip())
                if match:
                    signal_name = match.group(1)
                    num_bits = int(match.group(2))  # Number of bits
                    signal_value = match.group(3)  # Binary value

                    # If there's only 1 bit, do not add the bit-range
                    if num_bits == 1:
                        name = signal_name
                    else:
                        # Format the name with the bit-range (e.g., BL_NUM_TGT<[1:0]> for 2 bits)
                        name = "%s" % (signal_name)

                    # Format the binary value to match the number of bits
                    binary_value = signal_value.zfill(num_bits)
                    sc_signal[name] = binary_value


    # print(sc_signal)
    output_list = []
    for signal in SC_order:
        if "BLANK" in signal:
            num_blanks = re.findall(r'\d+', signal)
            output_list += [0]*int(num_blanks[0])
        else:
            temp = sc_signal[signal][::-1]
            output_list += list(map(int,temp))
    print(sc_signal["SAMPLE_EDGE_TIME"])
    output_list = output_list[::-1] + [0]*30
    return output_list

def set_CS(cs):
    ret_data = 0
    if cs == 0:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 1:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,0]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 2:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [0,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    if cs == 3:
        signal_array = [Utils.CS_0,Utils.CS_1]
        value_array = [1,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
    return ret_data

In [ ]:
#Call Scan IN here with data
scan_id = Utils.CONTROL_LOGIC_SC["id"]
scan_len = Utils.CONTROL_LOGIC_SC["len"]
scan_data = scan_gen(5)
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
#print(ret_data)

In [ ]:
#Initialise SRAM with default data.
write_data_default = Utils.get_Default_Write_Data().tolist()
ret_data = Utils.Write_SRAM(client_socket,write_data_default,1)
print(ret_data)

In [ ]:
#Turn on Analog PS
PS_Utils.set_voltage_E36313A(device_dc,ANA_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,ANA_LS))

In [ ]:
# #Setup VB Sweep Supply 
# Device_ID_VarPS = "USB0::0x2A8D::0x9201::MY61391394::0::INSTR"
# device_varps = PS_Utils.connect_B2902B(Device_ID_VarPS)
# PS_Utils.set_voltage_B2902B(device_varps,1,0.8,0.01)

In [ ]:
#SRAM IMC DATA
def SRAM_IMC_DATA(NUM_IN,Num_Zeros_Ref):
    arr = np.zeros((320, 1024),dtype=np.uint8)
    #Keep all the Delay lines at 1
    arr[8:][:] = 1  #0:7 are dedicated to Inputs.
    #Change Reflines to be at zero  
    for refline in range(24,280,36):
        #BANK 0
        Num_Zeros = Num_Zeros_Ref
        arr[refline,768:768+Num_Zeros] = 0 #BL3
        arr[refline+1,768:768+Num_Zeros] = 0 #BL2
        arr[refline+2,768:768+Num_Zeros] = 0 #BL1
        arr[refline+3,768:768+Num_Zeros] = 0 #BL0

        #BANK 1
        arr[refline,512:512+Num_Zeros] = 0 #BL3
        arr[refline+1,512:512+Num_Zeros] = 0 #BL2
        arr[refline+2,512:512+Num_Zeros] = 0 #BL1
        arr[refline+3,512:512+Num_Zeros] = 0 #BL0

        #BANK 2
        arr[refline,256:256+Num_Zeros] = 0 #BL3
        arr[refline+1,256:256+Num_Zeros] = 0 #BL2
        arr[refline+2,256:256+Num_Zeros] = 0 #BL1
        arr[refline+3,256:256+Num_Zeros] = 0 #BL0

        #BANK 3
        arr[refline,0:0+Num_Zeros] = 0 #BL3
        arr[refline+1,0:0+Num_Zeros] = 0 #BL2
        arr[refline+2,0:0+Num_Zeros] = 0 #BL1
        arr[refline+3,0:0+Num_Zeros] = 0 #BL0

    #Now give NUM_IN worth Input lines on.
    #Bank 3
    arr[4:8,0:0+NUM_IN] = 1
    arr[0:4,NUM_IN:256] = 1

    #Bank 2
    arr[4:8,256:256+NUM_IN] = 1
    arr[0:4,256+NUM_IN:512] = 1

    #Bank 1
    arr[4:8,512:512+NUM_IN] = 1
    arr[0:4,512+NUM_IN:768] = 1
    
    #Bank 0
    arr[4:8,768:768+NUM_IN] = 1
    arr[0:4,768+NUM_IN:1024] = 1
    arr = arr.flatten(order='F')
    return arr

In [ ]:
Delay_Data = []
for Input in range(0,257,1):
    write_data = SRAM_IMC_DATA(Input,256-Input).tolist()
    ret_data = Utils.Write_SRAM_Masked(client_socket,write_data,0)
    #print(ret_data)
    for DL in range(0,10):
        '''
        #Call DL EN Scan Chain here with data
        scan_id = Utils.DL_EN_SC_LM["id"]
        scan_len = Utils.DL_EN_SC_LM["len"]
        scan_data = np.zeros(scan_len,dtype=int).tolist()
        scan_data[DL] = 1
        ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,0)
        #print(ret_data)

        #Set IO to default and exit    
        ret_data = Utils.Set_ChipIO_to_Default(client_socket)
        #print(ret_data)

        signal_array = [Utils.BANK_EN_0,Utils.PRECHARGE,Utils.DEL_RST,Utils.TDC_RST,Utils.ENTIME,Utils.InputEN_DAC]
        value_array = [1,1,1,1,1,1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
        #print(ret_data)

        signal_array = [Utils.CALIB_EN]
        value_array = [1]
        ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
        #print(ret_data)
        '''
        Utils.Fast_Calib(client_socket,DL,0)
        temp = []
        for cp in range(5):
            time1 = scope.query(":MEASure:SETuptime? CHAN2,RISing,CHAN1,RISing")
            temp.append(float(time1))
        #print(time1)
        #time2 = scope.query(":MEASure:SETuptime? CHAN2,FALLing,CHAN1,FALLing")
        #print(time2)
        #delay = (float(time1)+float(time2))/2
        delay = np.mean(temp)
        Delay_Data.append(delay)
        #Set IO to default and exit    
        ret_data = Utils.Set_ChipIO_to_Default(client_socket)
        #print(ret_data)
        print(Input,DL,delay)

In [ ]:
import matplotlib.pyplot as plt
delay_diff = np.diff(Delay_Data)
plt.plot( delay_diff[:], marker='o', color='b', linestyle='-', label='y = x²')


In [ ]:
np.save(r"D:\Chip2025\Chip2025_Testing\Python_Notebook\Tests\Data\BANK3_CELL_DATA_MEAN.npy",Delay_Data)

In [ ]:
arr_write_default = write_data_default
arr_write = write_data
u8_array = []
print(len(arr_write_default))
k = 0
for i in range(0, len(arr_write_default), 8):
    if np.floor(i / 320) in Utils.calib_wl_idx:
        byte_bits = arr_write_default[i:i+8]
        value = sum(bit << j for j, bit in enumerate(byte_bits))  # index 0 is LSB
        u8_array.append(value)
    else:
        byte_bits = arr_write[k:k+8]
        value = sum(bit << j for j, bit in enumerate(byte_bits))  # index 0 is LSB
        u8_array.append(value)
        k = k+8

print(len(u8_array),k)

In [ ]:
#Call Read SRAM here
SRAM_DATA = Utils.Read_SRAM(client_socket,1)
print(len(SRAM_DATA))
print(SRAM_DATA)

In [ ]:
SRAM_DATA_Filtered = [ i for _,i in enumerate(SRAM_DATA) if (_ % 40) < 37]
u8_array_Filtered = [ i for _,i in enumerate(u8_array) if (_ % 40) < 37]
print((u8_array_Filtered))
print((SRAM_DATA_Filtered))
if SRAM_DATA_Filtered == u8_array_Filtered:
    print("Lists are equal")
else:
    print("Lists are not equal")

In [ ]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

In [ ]:
PS_Utils.kill_channel_E36313A(device_dc,ANA_LS)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,MAIN)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,DIG_LS)
PS_Utils.disconnect_E36313A(device_dc)
scope.close()